<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/Day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import time
assam_districts = {
    "Kamrup_Metropolitan": (26.1445, 91.7362), "Nalbari": (26.4430, 91.4426),
    "Barpeta": (26.3220, 91.0044), "Dhubri": (26.0207, 89.9743),
    "Goalpara": (26.1738, 90.6277), "Bongaigaon": (26.4751, 90.5540),
    "Kokrajhar": (26.4011, 90.2663), "Chirang": (26.6576, 90.5517),
    "Baksa": (26.6935, 91.5984), "Udalguri": (26.7455, 92.0963),
    "Darrang": (26.4428, 92.0298), "Sonitpur": (26.7644, 92.8368),
    "Biswanath": (26.7323, 93.1517), "Lakhimpur": (27.2354, 94.1042),
    "Dhemaji": (27.4815, 94.5574), "Morigaon": (26.2498, 92.3364),
    "Nagaon": (26.3475, 92.6840), "Hojai": (26.0028, 92.8523),
    "Karbi_Anglong": (26.0201, 93.5358), "West_Karbi_Anglong": (25.9000, 92.5300),
    "Dima_Hasao": (25.1764, 93.0232), "Cachar": (24.8333, 92.7789),
    "Hailakandi": (24.6806, 92.5647), "Karimganj": (24.8649, 92.3551),
    "Golaghat": (26.5160, 93.9688), "Jorhat": (26.7509, 94.2037),
    "Majuli": (26.9535, 94.1687), "Sivasagar": (26.9825, 94.6425),
    "Charaideo": (26.9317, 94.7397), "Dibrugarh": (27.4728, 94.9120),
    "Tinsukia": (27.4886, 95.3558), "South_Salmara": (25.7500, 89.9500),
    "Bajali": (26.5000, 91.0000), "Tamulpur": (26.6341, 91.5833),
    "Kamrup_Rural": (26.3197, 91.4402)
}
url = "https://archive-api.open-meteo.com/v1/archive"
all_districts_list = []

for district, (lat, lon) in assam_districts.items():
    print(f"📡 Fetching {district}...")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2021-01-01",
        "end_date": "2025-12-31",
        "hourly": "temperature_2m,rain,evapotranspiration,soil_moisture_0_to_7cm",
        "timezone": "Asia/Kolkata"
    }

    # Retry loop to handle minor network hiccups
    for attempt in range(3):
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                hourly = response.json()["hourly"]

                # Structure hourly matrix
                df_hourly = pd.DataFrame({
                    "Timestamp": pd.to_datetime(hourly["time"]),
                    "Temp": hourly["temperature_2m"],
                    "Rain": hourly["rain"],
                    "ET": hourly["evapotranspiration"],
                    "Soil": hourly["soil_moisture_0_to_7cm"]
                })

                # Aggregate to daily metrics
                df_hourly['Date'] = df_hourly['Timestamp'].dt.date
                df_daily = df_hourly.groupby('Date').agg(
                    Max_Temp_C=('Temp', 'max'),
                    Min_Temp_C=('Temp', 'min'),
                    Daily_Rain_mm=('Rain', 'sum'),
                    Evapotranspiration_mm=('ET', 'sum'),
                    Soil_Moisture_Pct=('Soil', 'mean')
                ).reset_index()

                df_daily['District'] = district
                all_districts_list.append(df_daily)
                break
            else:
                print(f"⚠️ Warning: Server returned status {response.status_code}, retrying...")
        except Exception as e:
            print(f"⚠️ Connection error on attempt {attempt+1}: {e}")
        time.sleep(2) # Rest between retries

    time.sleep(1) # Polite gap between districts to avoid getting IP blocked

# Combine all data frames into one master file
if all_districts_list:
    master_df = pd.concat(all_districts_list, ignore_index=True)
    master_df.to_csv('assam_raw_5year_data.csv', index=False)
    print("\n🎉 SUCCESS! Full statewide historical dataset compiled.")
    print(f"📊 Total Rows Gathered: {len(master_df)}")
else:
    print("❌ Fatal Error: Could not download data.")

📡 Fetching Kamrup_Metropolitan...
📡 Fetching Nalbari...
📡 Fetching Barpeta...
📡 Fetching Dhubri...
📡 Fetching Goalpara...
📡 Fetching Bongaigaon...
📡 Fetching Kokrajhar...
📡 Fetching Chirang...
📡 Fetching Baksa...
📡 Fetching Udalguri...
📡 Fetching Darrang...
📡 Fetching Sonitpur...
📡 Fetching Biswanath...
⚠️ Warning: Server returned status 429, retrying...
⚠️ Warning: Server returned status 429, retrying...
⚠️ Warning: Server returned status 429, retrying...
📡 Fetching Lakhimpur...
⚠️ Warning: Server returned status 429, retrying...
📡 Fetching Dhemaji...
📡 Fetching Morigaon...
📡 Fetching Nagaon...
📡 Fetching Hojai...
📡 Fetching Karbi_Anglong...
📡 Fetching West_Karbi_Anglong...
📡 Fetching Dima_Hasao...
📡 Fetching Cachar...
📡 Fetching Hailakandi...
📡 Fetching Karimganj...
📡 Fetching Golaghat...
📡 Fetching Jorhat...
⚠️ Warning: Server returned status 429, retrying...
⚠️ Warning: Server returned status 429, retrying...
⚠️ Warning: Server returned status 429, retrying...
📡 Fetching Majuli...


In [2]:
import requests
import pandas as pd
import time

print("🛡️ RUNNING TARGETED DATA PATCH FOR MISSING DISTRICTS...")

# Coordinates for the specific districts that hit the 429 rate limit
failed_districts = {
    "Biswanath": (26.7323, 93.1517),
    "Jorhat": (26.7509, 94.2037)
}

url = "https://archive-api.open-meteo.com/v1/archive"
patched_data_list = []

for district, (lat, lon) in failed_districts.items():
    print(f"📡 Retrying extraction for: {district}...")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2021-01-01",
        "end_date": "2025-12-31",
        "hourly": "temperature_2m,rain,evapotranspiration,soil_moisture_0_to_7cm",
        "timezone": "Asia/Kolkata"
    }

    # We increase to 5 attempts and add a longer cooldown sleep
    for attempt in range(5):
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                hourly = response.json()["hourly"]

                df_hourly = pd.DataFrame({
                    "Timestamp": pd.to_datetime(hourly["time"]),
                    "Temp": hourly["temperature_2m"],
                    "Rain": hourly["rain"],
                    "ET": hourly["evapotranspiration"],
                    "Soil": hourly["soil_moisture_0_to_7cm"]
                })

                df_hourly['Date'] = df_hourly['Timestamp'].dt.date
                df_daily = df_hourly.groupby('Date').agg(
                    Max_Temp_C=('Temp', 'max'),
                    Min_Temp_C=('Temp', 'min'),
                    Daily_Rain_mm=('Rain', 'sum'),
                    Evapotranspiration_mm=('ET', 'sum'),
                    Soil_Moisture_Pct=('Soil', 'mean')
                ).reset_index()

                df_daily['District'] = district
                patched_data_list.append(df_daily)
                print(f"✅ Successfully recovered data for {district}!")
                break
            elif response.status_code == 429:
                print(f"⚠️ Server is rate-limiting (429). Cooling down for 10 seconds... (Attempt {attempt+1}/5)")
                time.sleep(10)
        except Exception as e:
            print(f"⚠️ Connection glitch: {e}")
            time.sleep(5)

    # Explicit 5-second gap between districts to stay under the API radar
    time.sleep(5)

# Merge the missing pieces back into your main dataset
if patched_data_list:
    # 1. Load your existing imperfect dataset
    existing_df = pd.read_csv('assam_raw_5year_data.csv')

    # 2. Combine it with the recovered patches
    patch_df = pd.concat(patched_data_list, ignore_index=True)
    complete_df = pd.concat([existing_df, patch_df], ignore_index=True)

    # 3. Overwrite the file with the complete, fully repaired dataset
    complete_df.to_csv('assam_raw_5year_data.csv', index=False)
    print("\n📦 PATCH INTEGRATION COMPLETE!")
    print(f"📈 New Total Row Count: {len(complete_df)} rows across {complete_df['District'].nunique()}/35 districts.")
else:
    print("❌ Patch failed. The server is still blocking requests. Try running again in a couple minutes.")

🛡️ RUNNING TARGETED DATA PATCH FOR MISSING DISTRICTS...
📡 Retrying extraction for: Biswanath...
✅ Successfully recovered data for Biswanath!
📡 Retrying extraction for: Jorhat...
✅ Successfully recovered data for Jorhat!

📦 PATCH INTEGRATION COMPLETE!
📈 New Total Row Count: 63910 rows across 35/35 districts.


In [3]:
import pandas as pd

print("💧 Layer 2: Loading raw data and starting Hydrology calculations...")

df = pd.read_csv('assam_raw_5year_data.csv')

# 2. Define the Topographic Wetness Index (TWI) Multiplier
# Low values (0.4) for mountains, High values (1.5) for massive river basins
twi_registry = {
    'Kamrup_Metropolitan': 1.0, 'Nalbari': 1.3, 'Barpeta': 1.4, 'Dhubri': 1.4,
    'Goalpara': 1.2, 'Bongaigaon': 1.2, 'Kokrajhar': 1.1, 'Chirang': 1.1,
    'Baksa': 1.1, 'Udalguri': 1.1, 'Darrang': 1.3, 'Sonitpur': 1.2,
    'Biswanath': 1.2, 'Lakhimpur': 1.4, 'Dhemaji': 1.5, 'Morigaon': 1.4,
    'Nagaon': 1.3, 'Hojai': 1.2, 'Karbi_Anglong': 0.6, 'West_Karbi_Anglong': 0.6,
    'Dima_Hasao': 0.4, 'Cachar': 1.3, 'Hailakandi': 1.3, 'Karimganj': 1.3,
    'Golaghat': 1.1, 'Jorhat': 1.2, 'Majuli': 1.5, 'Sivasagar': 1.2,
    'Charaideo': 1.1, 'Dibrugarh': 1.3, 'Tinsukia': 1.2, 'South_Salmara': 1.4,
    'Bajali': 1.3, 'Tamulpur': 1.1, 'Kamrup_Rural': 1.2
}
df['TWI_Multiplier'] = df['District'].map(twi_registry)

# 3. Calculate Soil Absorption capacity
# We assume the top 7cm of soil can swallow up to 20mm of water before pooling.
# If Soil_Moisture_Pct is 1.0 (100% full), absorption is 0mm.
MAX_SOIL_CAPACITY_MM = 20.0
df['Soil_Absorption_mm'] = MAX_SOIL_CAPACITY_MM * (1.0 - df['Soil_Moisture_Pct'])

# 4. Calculate Net Standing Water (Rain minus what the Sun evaporates minus what the Earth drinks)
# .clip(lower=0) ensures that if evaporation/absorption is bigger than rain, we get 0, not a negative number.
df['Net_Standing_Water_mm'] = (
    df['Daily_Rain_mm'] - df['Evapotranspiration_mm'] - df['Soil_Absorption_mm']
).clip(lower=0)

# 5. Apply the Topographic landscape modifier to calculate final vector breeding sites
df['Final_Breeding_Water_mm'] = df['Net_Standing_Water_mm'] * df['TWI_Multiplier']

# Save this progress to an intermediate file so we don't lose it
df.to_csv('assam_processed_layer2.csv', index=False)

print("📝 Done! Let's check a slice of our new simulated physics:")
# Let's inspect a flat basin (Majuli) vs a mountain zone (Dima Hasao) on a rainy day
sample = df[df['Daily_Rain_mm'] > 5][['District', 'Daily_Rain_mm', 'Soil_Absorption_mm', 'TWI_Multiplier', 'Final_Breeding_Water_mm']].head(10)
print(sample.to_string())

💧 Layer 2: Loading raw data and starting Hydrology calculations...
📝 Done! Let's check a slice of our new simulated physics:
                District  Daily_Rain_mm  Soil_Absorption_mm  TWI_Multiplier  Final_Breeding_Water_mm
18   Kamrup_Metropolitan           12.5           11.655833             1.0                 0.844167
89   Kamrup_Metropolitan           11.5           11.471667             1.0                 0.028333
105  Kamrup_Metropolitan            8.8           13.910000             1.0                 0.000000
107  Kamrup_Metropolitan           10.5           11.413333             1.0                 0.000000
121  Kamrup_Metropolitan           11.1           13.150000             1.0                 0.000000
122  Kamrup_Metropolitan            6.7           11.354167             1.0                 0.000000
124  Kamrup_Metropolitan           12.4           10.848333             1.0                 1.551667
127  Kamrup_Metropolitan           17.0           10.525000        

In [4]:
import pandas as pd
df = pd.read_csv('assam_raw_5year_data.csv')
print(f"Total rows in raw data: {len(df)}")
print(f"Total districts captured: {df['District'].nunique()}/35")

Total rows in raw data: 63910
Total districts captured: 35/35


In [5]:
import pandas as pd

print("💧 Generating complete statewide Hydrology Engine Matrix...")

# 1. Load the raw dataset (Make sure it contains all patched districts!)
df = pd.read_csv('assam_raw_5year_data.csv')

# 2. Complete Topographic Wetness Index (TWI) Registry for all 35 districts
twi_registry = {
    'Kamrup_Metropolitan': 1.0, 'Nalbari': 1.3, 'Barpeta': 1.4, 'Dhubri': 1.4,
    'Goalpara': 1.2, 'Bongaigaon': 1.2, 'Kokrajhar': 1.1, 'Chirang': 1.1,
    'Baksa': 1.1, 'Udalguri': 1.1, 'Darrang': 1.3, 'Sonitpur': 1.2,
    'Biswanath': 1.2, 'Lakhimpur': 1.4, 'Dhemaji': 1.5, 'Morigaon': 1.4,
    'Nagaon': 1.3, 'Hojai': 1.2, 'Karbi_Anglong': 0.6, 'West_Karbi_Anglong': 0.6,
    'Dima_Hasao': 0.4, 'Cachar': 1.3, 'Hailakandi': 1.3, 'Karimganj': 1.3,
    'Golaghat': 1.1, 'Jorhat': 1.2, 'Majuli': 1.5, 'Sivasagar': 1.2,
    'Charaideo': 1.1, 'Dibrugarh': 1.3, 'Tinsukia': 1.2, 'South_Salmara': 1.4,
    'Bajali': 1.3, 'Tamulpur': 1.1, 'Kamrup_Rural': 1.2
}
df['TWI_Multiplier'] = df['District'].map(twi_registry)

# 3. Calculate dynamic Soil Absorption capacity (Max 20mm pool cap)
MAX_SOIL_CAPACITY_MM = 20.0
df['Soil_Absorption_mm'] = MAX_SOIL_CAPACITY_MM * (1.0 - df['Soil_Moisture_Pct'])

# 4. Net Standing Water Equation (Clamped at 0 to avoid negative water math)
df['Net_Standing_Water_mm'] = (
    df['Daily_Rain_mm'] - df['Evapotranspiration_mm'] - df['Soil_Absorption_mm']
).clip(lower=0)

# 5. Generate final breeding landscape calculation
df['Final_Breeding_Water_mm'] = df['Net_Standing_Water_mm'] * df['TWI_Multiplier']

# 6. Save the final statewide dataset
df.to_csv('assam_processed_layer2_complete.csv', index=False)

print("\n🎉 SUCCESS! Complete Layer 2 Dataset Updated.")
print(f"📊 Matrix Dimensions: {df.shape} (Rows, Columns)")
print("🔍 Checking row distribution per unique district to ensure perfect balance:")
print(df['District'].value_counts())

💧 Generating complete statewide Hydrology Engine Matrix...

🎉 SUCCESS! Complete Layer 2 Dataset Updated.
📊 Matrix Dimensions: (63910, 11) (Rows, Columns)
🔍 Checking row distribution per unique district to ensure perfect balance:
District
Kamrup_Metropolitan    1826
Nalbari                1826
Barpeta                1826
Dhubri                 1826
Goalpara               1826
Bongaigaon             1826
Kokrajhar              1826
Chirang                1826
Baksa                  1826
Udalguri               1826
Darrang                1826
Sonitpur               1826
Lakhimpur              1826
Dhemaji                1826
Morigaon               1826
Nagaon                 1826
Hojai                  1826
Karbi_Anglong          1826
West_Karbi_Anglong     1826
Dima_Hasao             1826
Cachar                 1826
Hailakandi             1826
Karimganj              1826
Golaghat               1826
Majuli                 1826
Sivasagar              1826
Charaideo              1826
Dibrug